In [ ]:
# 第一个单元格：数据预处理（修改版，实现patientwise标准化）
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import h5py
import scipy.io
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

# 设置随机种子确保可重现性
torch.manual_seed(42)
torch.cuda.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# 设置设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 方法1: 使用相对路径（推荐）
export_path = './outputs/'  # 当前目录下的outputs文件夹

# 确保输出目录存在
import os
os.makedirs(export_path, exist_ok=True)
f = h5py.File('TRAIN38.mat','r')
arrays = {}
for k, v in f.items():
    arrays[k] = np.array(v)
f.close()
train_data = arrays['data'].transpose()
train_region = arrays['region'].transpose()
prob_idx = arrays['prob_idx'].transpose()
del arrays, f

# Patientwise标准化函数
def patientwise_standardize(data, patient_idx):
    """
    对每个patient的体素分别进行标准化
    
    参数:
    data: shape (N, 341) 的数据
    patient_idx: shape (N,) 的patient索引
    
    返回:
    standardized_data: 标准化后的数据，shape不变
    patient_stats: 字典，保存每个patient每个feature的均值和标准差
    """
    # 确保patient_idx是一维的
    if patient_idx.ndim > 1:
        patient_idx = patient_idx.ravel()
    
    # 确保数据类型是float，避免整数除法问题
    data = data.astype(np.float64)
    standardized_data = np.zeros_like(data)
    patient_stats = {}
    
    # 获取所有唯一的patient索引
    unique_patients = np.unique(patient_idx)
    
    print(f"数据形状: {data.shape}")
    print(f"Patient索引形状: {patient_idx.shape}")
    print(f"唯一的patient数量: {len(unique_patients)}")
    
    for patient_id in unique_patients:
        # 找到属于当前patient的所有体素
        patient_mask = (patient_idx == patient_id)
        patient_voxels = data[patient_mask]
        
        print(f"Patient {patient_id}: {patient_voxels.shape[0]} 个体素")
        
        # 计算该patient每个feature的均值和标准差
        patient_mean = np.mean(patient_voxels, axis=0)
        patient_std = np.std(patient_voxels, axis=0)
        
        # 避免除以0
        patient_std[patient_std == 0] = 1.0
        
        # 保存统计量
        patient_stats[patient_id] = {
            'mean': patient_mean,
            'std': patient_std
        }
        
        # 标准化该patient的数据
        standardized_data[patient_mask] = (patient_voxels - patient_mean) / patient_std
    
    return standardized_data, patient_stats

# 应用patientwise标准化的函数（用于测试集）
def apply_patientwise_standardize(data, patient_idx, patient_stats):
    """
    使用已有的统计量对新数据进行标准化
    
    参数:
    data: shape (N, 341) 的数据
    patient_idx: shape (N,) 的patient索引
    patient_stats: 包含每个patient统计量的字典
    
    返回:
    standardized_data: 标准化后的数据
    """
    standardized_data = np.zeros_like(data)
    
    unique_patients = np.unique(patient_idx)
    
    for patient_id in unique_patients:
        patient_mask = (patient_idx == patient_id)
        patient_voxels = data[patient_mask]
        
        if patient_id in patient_stats:
            # 使用已有的统计量
            patient_mean = patient_stats[patient_id]['mean']
            patient_std = patient_stats[patient_id]['std']
        else:
            # 如果是新的patient，计算其自己的统计量
            patient_mean = np.mean(patient_voxels, axis=0)
            patient_std = np.std(patient_voxels, axis=0)
            patient_std[patient_std == 0] = 1.0
        
        standardized_data[patient_mask] = (patient_voxels - patient_mean) / patient_std
    
    return standardized_data

# 确保prob_idx是一维数组
if prob_idx.ndim > 1:
    prob_idx = prob_idx.ravel()  # 或者使用 prob_idx.squeeze() 如果只有一个维度是1

# 分离训练集和验证集
curr_set = np.where(prob_idx != 38)[0]
set_data = train_data[curr_set,:]
set_region = train_region[curr_set,:]
set_prob_idx = prob_idx[curr_set]  # 保留训练集的patient索引
print(f"训练集数据形状: {set_data.shape}")
print(f"训练集patient索引形状: {set_prob_idx.shape}")

curr_val = np.where(prob_idx == 38)[0]
val_data = train_data[curr_val,:]
val_label = train_region[curr_val,:]
val_prob_idx = prob_idx[curr_val]  # 保留验证集的patient索引
print(f"验证集数据形状: {val_data.shape}")
print(f"验证集patient索引形状: {val_prob_idx.shape}")

del curr_set, curr_val, train_data, train_region

# 进一步分割训练集
X_train1, X_train2, y_train1, y_train2, idx_train1, idx_train2 = train_test_split(
    set_data, set_region, set_prob_idx, test_size=0.01, random_state=42
)
del set_data, set_region, set_prob_idx

# 对训练集进行patientwise标准化
print("正在进行训练集的patientwise标准化...")
X_train1, train_patient_stats = patientwise_standardize(X_train1, idx_train1)

# 对验证集进行patientwise标准化（使用验证集自己的统计量）
print("正在进行验证集的patientwise标准化...")
val_data, val_patient_stats = patientwise_standardize(val_data, val_prob_idx)

# 保存统计量以备后用（如果需要）
# 确保路径正确
save_path = export_path if export_path.endswith('/') else export_path + '/'
np.save(save_path + 'train_patient_stats.npy', train_patient_stats)
np.save(save_path + 'val_patient_stats.npy', val_patient_stats)

print("Patientwise标准化完成！")
print(f"训练集唯一patient数: {len(np.unique(idx_train1))}")
print(f"验证集唯一patient数: {len(np.unique(val_prob_idx))}")


In [ ]:


# ========================================
# 第二个单元格：训练模型（保持不变）
# 模型配置
batch_size = 128
no_epochs = 25
no_classes = 102

# L2正则化 - 只对权重矩阵，不包括偏置（完全模拟TensorFlow的kernel_regularizer）
def kernel_l2_regularization(model, weight_decay=0.00001):
    l2_reg = 0
    for name, param in model.named_parameters():
        # 只对权重矩阵应用L2正则化，跳过偏置项
        if 'weight' in name and param.requires_grad:
            l2_reg += torch.norm(param, p=2) ** 2
    return weight_decay * l2_reg

# 定义模型（严格对应TensorFlow版本）
class RegModel(nn.Module):
    def __init__(self, input_dim=341, num_classes=102):
        super(RegModel, self).__init__()
        # 对应TensorFlow的Dense层，使用默认初始化
        self.fc1 = nn.Linear(input_dim, 4096)
        self.fc2 = nn.Linear(4096, 4096) 
        self.fc3 = nn.Linear(4096, 4096)
        self.fc4 = nn.Linear(4096, 4096)
        self.fc5 = nn.Linear(4096, num_classes)  # visualized_layer对应的层
        self.dropout = nn.Dropout(0.5)
        
    def forward(self, x):
        # 严格按照TensorFlow模型的结构
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.dropout(F.relu(self.fc2(x)))
        x = self.dropout(F.relu(self.fc3(x)))
        x = self.dropout(F.relu(self.fc4(x)))
        x = self.fc5(x)  # 注意：这里不应用softmax，让CrossEntropyLoss处理
        return x

def create_reg_model():
    input_shape = [341]
    model = RegModel().to(device)
    return model

model = create_reg_model()

# 转换为PyTorch张量
X_train1_tensor = torch.FloatTensor(X_train1).to(device)
y_train1_tensor = torch.FloatTensor(y_train1).to(device)
val_data_tensor = torch.FloatTensor(val_data).to(device)
val_label_tensor = torch.FloatTensor(val_label).to(device)

# 创建数据加载器
train_dataset = TensorDataset(X_train1_tensor, y_train1_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# 编译模型（严格对应TensorFlow的配置）
optimizer = optim.Adam(model.parameters(), lr=0.00001)
criterion = nn.CrossEntropyLoss()

# 训练历史记录（对应TensorFlow的history）
history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': [], 
           'train_f1': [], 'val_f1': []}

# 训练循环（对应model.fit）
print("开始训练...")
for epoch in range(no_epochs):
    # 训练阶段
    model.train()
    epoch_train_loss = 0
    correct_train = 0
    total_train = 0
    all_train_preds = []
    all_train_labels = []
    
    for batch_idx, (data, target) in enumerate(train_loader):
        optimizer.zero_grad()
        
        output = model(data)
        # 将one-hot编码转换为类别索引
        target_indices = torch.argmax(target, dim=1)
        
        # 计算基础损失
        base_loss = criterion(output, target_indices)
        
        # 添加L2正则化（只对权重，模拟kernel_regularizer）
        l2_reg = kernel_l2_regularization(model, weight_decay=0.00001)
        total_loss = base_loss + l2_reg
        
        total_loss.backward()
        optimizer.step()
        
        epoch_train_loss += total_loss.item()
        
        # 计算训练准确率
        _, predicted = torch.max(output.data, 1)
        total_train += target_indices.size(0)
        correct_train += (predicted == target_indices).sum().item()
        
        # 收集预测结果用于计算F1
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_labels.extend(target_indices.cpu().numpy())
    
    # 验证阶段
    model.eval()
    with torch.no_grad():
        val_output = model(val_data_tensor)
        val_target_indices = torch.argmax(val_label_tensor, dim=1)
        val_base_loss = criterion(val_output, val_target_indices)
        val_l2_reg = kernel_l2_regularization(model, weight_decay=0.00001)
        val_total_loss = val_base_loss + val_l2_reg
        
        # 计算验证准确率
        _, val_predicted = torch.max(val_output.data, 1)
        val_correct = (val_predicted == val_target_indices).sum().item()
        val_accuracy = val_correct / val_target_indices.size(0)
        
        # 计算验证集（38号病人）的预测结果
        val_preds_numpy = val_predicted.cpu().numpy()
        val_labels_numpy = val_target_indices.cpu().numpy()
    
    # 计算macro F1分数
    train_f1 = f1_score(all_train_labels, all_train_preds, average='macro', zero_division=0)
    val_f1 = f1_score(val_labels_numpy, val_preds_numpy, average='macro', zero_division=0)
    
    # 记录历史
    avg_train_loss = epoch_train_loss / len(train_loader)
    train_accuracy = correct_train / total_train
    
    history['loss'].append(avg_train_loss)
    history['val_loss'].append(val_total_loss.item())
    history['accuracy'].append(train_accuracy)
    history['val_accuracy'].append(val_accuracy)
    history['train_f1'].append(train_f1)
    history['val_f1'].append(val_f1)
    
    # 打印每个epoch的结果
    print(f'Epoch [{epoch+1}/{no_epochs}] - '
          f'Train Loss: {avg_train_loss:.4f}, '
          f'Train Acc: {train_accuracy:.4f}, '
          f'Train F1: {train_f1:.4f}, '
          f'Val Loss: {val_total_loss.item():.4f}, '
          f'Val Acc: {val_accuracy:.4f}, '
          f'Val F1 (Patient 38): {val_f1:.4f}')

print("\n训练完成！")
print(f"最佳验证准确率: {max(history['val_accuracy']):.4f}")
print(f"最佳验证F1分数 (Patient 38): {max(history['val_f1']):.4f}")

# 保存模型
torch.save(model.state_dict(), export_path + 'dense_4x4096_model.pth')


In [ ]:
# ========================================
# 第三个单元格：应用模型（修改版，应用patientwise标准化）
filepath = 'DEMO38.mat'
arrays = {}
f = h5py.File(filepath,'r')
for k, v in f.items():
    arrays[k] = np.array(v)
f.close()
multidim_data = arrays['multidim_data'].transpose()
# 假设DEMO38.mat中也有prob_idx，如果没有需要其他方式获取
if 'prob_idx' in arrays:
    demo_prob_idx = arrays['prob_idx'].transpose()
else:
    # 如果没有prob_idx，可能需要用其他方式确定patient索引
    # 这里假设所有数据来自同一个patient
    demo_prob_idx = np.zeros(multidim_data.shape[0], dtype=int)
del arrays, f

# 对测试数据进行patientwise标准化
# 如果测试数据的patient在训练集中出现过，使用训练集的统计量
# 否则使用测试数据自己的统计量
print("正在对测试数据进行patientwise标准化...")
multidim_data = apply_patientwise_standardize(multidim_data, demo_prob_idx, train_patient_stats)

# 预测（对应model.predict）
multidim_data_tensor = torch.FloatTensor(multidim_data).to(device)
model.eval()
with torch.no_grad():
    predicted_ann = model(multidim_data_tensor)
    # TensorFlow模型最后一层有softmax，我们需要手动应用
    predicted_ann = F.softmax(predicted_ann, dim=1)
    predicted_ann = predicted_ann.cpu().numpy()

predictpath = 'Your predictpath'
scipy.io.savemat(predictpath +'dense_4x4096_model_prediction.mat', mdict={'predicted_ann':predicted_ann})


In [ ]:
# ========================================
# 第四个单元格：计算和可视化显著性（修改版，考虑patientwise标准化）
# 注意：由于无法完全复制keras-vis的实现，这部分结果可能略有差异
import matplotlib.pyplot as plt

# PyTorch版本的显著性计算（尽可能模拟keras-vis）
def visualize_saliency(model, layer_index, filter_indices, seed_input, keepdims=True):
    """
    尽可能模拟keras-vis的visualize_saliency函数
    注意：由于内部实现差异，结果可能与keras-vis略有不同
    """
    model.eval()
    
    # 确保输入是正确的形状
    if seed_input.ndim == 1:
        seed_input = seed_input.reshape(1, -1)
    elif seed_input.ndim == 3:
        # 如果是三维，flatten到二维
        seed_input = seed_input.reshape(1, -1)
    
    seed_input_tensor = torch.FloatTensor(seed_input).to(device)
    seed_input_tensor.requires_grad_(True)
    
    output = model(seed_input_tensor)
    
    # 对目标类别计算梯度
    target_score = output[0, filter_indices]
    
    # 计算梯度
    model.zero_grad()
    target_score.backward(retain_graph=True)
    
    # 返回输入梯度的绝对值作为显著性
    saliency = seed_input_tensor.grad.data.abs()
    
    if keepdims:
        return saliency.cpu().numpy()
    else:
        return saliency.squeeze().cpu().numpy()

# 模拟utils.find_layer_idx
def find_layer_idx(model, layer_name):
    if layer_name == 'visualized_layer':
        return len(list(model.children())) - 1
    return -1

# 反标准化函数
def inverse_patientwise_standardize(data, patient_idx, patient_stats):
    """
    将标准化的数据还原到原始尺度
    """
    inverse_data = np.zeros_like(data)
    
    for i in range(len(data)):
        patient_id = patient_idx[i]
        if patient_id in patient_stats:
            mean = patient_stats[patient_id]['mean']
            std = patient_stats[patient_id]['std']
            inverse_data[i] = data[i] * std + mean
        else:
            # 如果没有统计量，返回原始数据
            inverse_data[i] = data[i]
    
    return inverse_data

# 要可视化的数字
indices_to_visualize = [ 10000, 20000, 30000, 40000, 50000, 60000 ] #or range(0,173383)
layer_index = find_layer_idx(model, 'visualized_layer')

# 可视化
for index_to_visualize in indices_to_visualize:
    # 获取输入 - 修正维度问题
    input_spectrum = val_data[index_to_visualize,:]  # val_data是2D数组，使用2D索引
    input_class = np.argmax(val_label[index_to_visualize,:])
    print(str(input_class))
    
    # 生成可视化
    visualization = visualize_saliency(model, layer_index, filter_indices=input_class, seed_input=input_spectrum, keepdims=True)
    
    # 反标准化以显示原始尺度
    patient_id = val_prob_idx[index_to_visualize]
    if patient_id in val_patient_stats:
        original_spectrum = inverse_patientwise_standardize(
            input_spectrum.reshape(1, -1), 
            np.array([patient_id]), 
            val_patient_stats
        ).flatten()
    else:
        original_spectrum = input_spectrum
    
    plt.figure()
    plt.title(str(input_class))
    plt.plot(visualization.flatten(),'g', label='Saliency')
    plt.plot(original_spectrum, alpha=0.25, label='Original spectrum')
    plt.plot(input_spectrum,'b',alpha=0.3, label='Standardized spectrum')
    plt.axvspan(0, 15, color='gray', alpha=0.3)
    plt.axvspan(225, 230, color='gray', alpha=0.2)
    plt.legend()
    plt.show()
